In [1]:
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests

d:\Clg Kabir_\Gen-AI-\Gen-AI-\LangChain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm=HuggingFaceEndpoint(model='Qwen/Qwen3-32B',
                        task='text_generation')
model=ChatHuggingFace(llm=llm)


In [ ]:
@tool
def conversion_factor(base_currency:str , target_currency:str)->float:
    """This Function fetches the currency conversion factor between base currency and target currency """
    url=f'https://v6.exchangerate-api.com/v6/9e7ee1833f687acf8ec5/pair/{base_currency}/{target_currency}'
    response=requests.get(url)
    return response.json()

@tool
def convert(base_currency_value:float , conversion_rate: Annotated[float, InjectedToolArg] )-> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base   currency value
    """
    return base_currency_value * conversion_rate

In [5]:
conversion_factor.invoke({"base_currency":"USD","target_currency" :"INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1783555202,
 'time_last_update_utc': 'Thu, 09 Jul 2026 00:00:02 +0000',
 'time_next_update_unix': 1783641602,
 'time_next_update_utc': 'Fri, 10 Jul 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.6063}

In [6]:
convert.invoke({"base_currency_value":100 ,"conversion_rate":95.6063} )

9560.630000000001

In [7]:
llm_with_tools=model.bind_tools([conversion_factor, convert])

In [8]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [9]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [10]:
ai_message = llm_with_tools.invoke(messages)

In [11]:
messages.append(ai_message)
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'conversion_factor', 'description': None}, 'id': '1zx0avaj3', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert', 'description': None}, 'id': '508wgvjt8', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 481, 'prompt_tokens': 268, 'total_tokens': 749}, 'model_name': 'Qwen/Qwen3-32B', 'system_fingerprint': 'fp_d58dbe76cd', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f469e-9eac-7813-ae38-e80418222ecb-0', tool_calls=[{'name': 'conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '1zx0avaj3', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency

In [12]:
ai_message.tool_calls

[{'name': 'conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': '1zx0avaj3',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '508wgvjt8',
  'type': 'tool_call'}]

In [13]:
import json 
for tool_call in ai_message.tool_calls:
    # Executing 1st tool and getting value of conversion factor
    if tool_call['name']=='conversion_factor':
        tool_message1=conversion_factor.invoke(tool_call)
        # fetch the conversion factor
        conversion_rate=json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to message list
        messages.append(tool_message1)
    # Execute the 2nd tool using the conversion factor from tool 1
    if tool_call['name']=='convert':
        #fetch current args
        tool_call['args']['conversion_rate']=conversion_rate
        tool_message2=convert.invoke(tool_call)
        messages.append(tool_message2)        

In [14]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'conversion_factor', 'description': None}, 'id': '1zx0avaj3', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert', 'description': None}, 'id': '508wgvjt8', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 481, 'prompt_tokens': 268, 'total_tokens': 749}, 'model_name': 'Qwen/Qwen3-32B', 'system_fingerprint': 'fp_d58dbe76cd', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f469e-9eac-7813-ae38-e80418222ecb-0', tool_calls=[{'name': 'conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '1zx0avaj3', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency

In [16]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is **0.01046**, based on data last updated on **2026-07-09 00:00:02 UTC**.  \n\nConverting **10 INR** to USD:  \n**10 INR × 0.01046 = 0.1046 USD**.  \n\nFor reference, the next rate update is scheduled on **2026-07-10 00:00:02 UTC**. Let me know if you need further details!  \n\n*(Documentation: [exchangerate-api.com/docs](https://www.exchangerate-api.com/docs), Terms of use: [exchangerate-api.com/terms](https://www.exchangerate-api.com/terms))*'